# 08 — Freshness

## Goal
Determine whether a zone has been **touched before** — untouched zones are significantly stronger because the orders that created them have not been partially filled yet.

---

## Definition

A zone is **touched** every time a candle's close re-enters the zone box after the base.  
A zone **dies** (is invalidated) when a candle closes **beyond the distal line** — the structure has been broken.

| Touches after formation | Freshness score |
|-------------------------|-----------------|
| 0 (never re-entered)    | 2 |
| 1                       | 1 |
| ≥ 2                     | 0 |

---

## Algorithm

For each confirmed zone, scan every candle **after** `be + DEPARTURE_CANDLES`:

$$\text{touch} = \begin{cases}
+1 & \text{if } \text{distal} \leq \text{close} \leq \text{proximal} \quad (\text{demand}) \\
+1 & \text{if } \text{proximal} \leq \text{close} \leq \text{distal} \quad (\text{supply}) \\
\text{dead} & \text{if close beyond distal}
\end{cases}$$

Once dead, scanning stops — all subsequent touches would belong to a different price cycle.

---

## Visualization
Each zone rectangle is annotated with its freshness score (0 / 1 / 2) and colour-coded by score intensity.


## 1. Load data and run the detection pipeline

In [ ]:
import pandas as pd
import numpy as np

# ── constants ────────────────────────────────────────────────────────────────
BASE_BODY_RATIO_MAX  = 0.50
BASE_MIN_CANDLES     = 1
BASE_MAX_CANDLES     = 5
BASE_MAX_ATR_WIDTH   = 2.5
BASE_COMPACTNESS_MAX = 0.80
LEG_CANDLES          = 3
LEG_ATR_MIN          = 1.5
DEPARTURE_CANDLES    = 3
DEPARTURE_ATR_MIN    = 1.5
DEPARTURE_RATIO_MIN  = 1.0

FORMATION_MAP = {
    ("up",   "up"):   ("RBR", "demand"),
    ("down", "down"): ("DBD", "supply"),
    ("down", "up"):   ("DBR", "demand"),
    ("up",   "down"): ("RBD", "supply"),
}

df      = pd.read_csv("../fixtures_enhanced.csv", index_col=0, parse_dates=True)
labeled = pd.read_csv("../fixtures_labeled.csv",  index_col=0, parse_dates=True)
o_arr = df["open"].values; h_arr = df["high"].values
l_arr = df["low"].values;  c_arr = df["close"].values; atr_arr = df["atr"].values

def find_base_clusters():
    clusters, i = [], 0
    while i < len(df):
        if not df["is_base"].iloc[i]: i += 1; continue
        j = i
        while j + 1 < len(df) and df["is_base"].iloc[j + 1]: j += 1
        if BASE_MIN_CANDLES <= (j - i + 1) <= BASE_MAX_CANDLES: clusters.append((i, j))
        i = j + 1
    return clusters

def cluster_ok(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    width   = h_arr[bs:be+1].max() - l_arr[bs:be+1].min()
    compact = (c_arr[bs:be+1].max() - c_arr[bs:be+1].min()) / width if width > 0 else 1
    return (width <= BASE_MAX_ATR_WIDTH * avg_atr) and (compact <= BASE_COMPACTNESS_MAX)

def classify_move(net, avg_atr):
    t = LEG_ATR_MIN * avg_atr
    return "up" if net >= t else ("down" if net <= -t else "flat")

def measure_legs(bs, be):
    avg_atr = atr_arr[bs:be+1].mean()
    if bs < LEG_CANDLES or be + LEG_CANDLES >= len(c_arr): return None
    return (classify_move(c_arr[bs-1] - o_arr[bs-LEG_CANDLES], avg_atr),
            classify_move(c_arr[be+LEG_CANDLES] - c_arr[be],   avg_atr),
            avg_atr)

def proximal_distal(bs, be, zone_type):
    bh, bl = h_arr[bs:be+1], l_arr[bs:be+1]
    return (bh.max(), bl.min()) if zone_type == "demand" else (bl.min(), bh.max())

def check_departure(proximal, be, zone_type, zone_width, avg_atr):
    end = min(len(c_arr)-1, be + DEPARTURE_CANDLES)
    ch  = h_arr[be+1:end+1]; cl = l_arr[be+1:end+1]
    if len(ch) == 0: return 0, 0, 0, False
    dep = (ch.max() - proximal) if zone_type == "demand" else (proximal - cl.min())
    dr  = dep / zone_width if zone_width > 0 else 0
    da  = dep / avg_atr   if avg_atr   > 0 else 0
    return round(dep,3), round(dr,2), round(da,2), (dr >= DEPARTURE_RATIO_MIN and da >= DEPARTURE_ATR_MIN)

def detect_zones():
    results = []
    for bs, be in find_base_clusters():
        if not cluster_ok(bs, be): continue
        legs = measure_legs(bs, be)
        if legs is None: continue
        lin_dir, lout_dir, avg_atr = legs
        pair = FORMATION_MAP.get((lin_dir, lout_dir))
        if pair is None: continue
        formation, zone_type = pair
        prox, dist = proximal_distal(bs, be, zone_type)
        zw = abs(prox - dist)
        dep, dep_ratio, dep_atr, passed = check_departure(prox, be, zone_type, zw, avg_atr)
        if not passed: continue
        results.append(dict(
            bs=bs, be=be, formation=formation, zone_type=zone_type,
            proximal=prox, distal=dist, zone_width=zw,
            departure=dep, dep_ratio=dep_ratio, dep_atr=dep_atr,
        ))
    return results

zones = detect_zones()
print(f"{len(zones)} zones detected")


## 2. `count_touches` — scan forward for re-entries

In [ ]:
FRESHNESS_SCORE = {0: 2, 1: 1}   # 2+ touches → 0

def count_touches(zone):
    prox, dist, zt = zone["proximal"], zone["distal"], zone["zone_type"]
    start = zone["be"] + DEPARTURE_CANDLES + 1
    touches = 0
    for i in range(start, len(c_arr)):
        close = c_arr[i]
        # zone death — closed beyond distal
        if zt == "demand" and close < dist: break
        if zt == "supply" and close > dist: break
        # touch — closed inside the zone box
        if zt == "demand" and dist <= close <= prox: touches += 1
        if zt == "supply" and prox <= close <= dist: touches += 1
    return touches

for z in zones:
    t = count_touches(z)
    z["touches"]         = t
    z["freshness_score"] = FRESHNESS_SCORE.get(t, 0)
    z["scenario"]        = labeled["scenario"].iloc[z["bs"]]

print(f"{'Scenario':<22} {'Type':8} {'Touches':>7} {'Score':>5}")
print("-" * 48)
for z in zones:
    print(f"{z['scenario']:<22} {z['zone_type']:8} {z['touches']:>7} {z['freshness_score']:>5}")


## 3. Visualization — freshness score on each zone

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"

COLOR_BULL = "#26a69a"
COLOR_BEAR = "#ef5350"
BG         = "#131722"
GRID       = "#1e222d"
EDGE       = {"demand": "#26a69a", "supply": "#ef5350"}

def fresh_fill(zone_type, score):
    base = (38, 166, 154) if zone_type == "demand" else (239, 83, 80)
    alpha = 0.05 + 0.10 * score   # 0→0.05, 1→0.15, 2→0.25
    return f"rgba({base[0]},{base[1]},{base[2]},{alpha})"

fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df.index, open=df["open"], high=df["high"],
    low=df["low"], close=df["close"],
    increasing_line_color=COLOR_BULL, decreasing_line_color=COLOR_BEAR, name="Price",
))

for z in zones:
    x0, x1 = df.index[z["bs"]], df.index[z["be"]]
    c_edge  = EDGE[z["zone_type"]]
    fig.add_vrect(x0=x0, x1=x1,
                  fillcolor=fresh_fill(z["zone_type"], z["freshness_score"]),
                  line=dict(color=c_edge, width=1, dash="dot"))
    fig.add_annotation(x=x0, y=z["proximal"],
                       text=f"{z['formation']} fresh={z['freshness_score']}",
                       showarrow=False, font=dict(size=8, color=c_edge),
                       xanchor="left", yanchor="bottom")

fig.update_layout(
    title="Freshness Score per Zone",
    height=560, plot_bgcolor=BG, paper_bgcolor=BG, font_color="white",
    xaxis_rangeslider_visible=False, hovermode="x unified",
    xaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False),
    yaxis=dict(gridcolor=GRID, showgrid=True, zeroline=False, title="Price"),
)
fig.show()
